In [1]:
!pip install sqlalchemy==1.4.54 sqlalchemy-utils==0.38.3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.3/100.3 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: sqlalchemy
    Found existing installation: SQLAlchemy 2.0.44
    Uninstalling SQLAlchemy-2.0.44:
      Successfully uninstalled SQLAlchemy-2.0.44
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.19.0 requires sqlalchemy<3.0.0,>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.
ipython-sql 0.5.0 requires sqlalchemy>=2.0, but you have sqlalchemy 1.4.54 which is incompatible.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

# set DB path inside your Drive
DB_PATH = '/content/drive/MyDrive/library.db'


Mounted at /content/drive


In [13]:
# Cell 3: Models + SQLAlchemy setup + helper functions
from sqlalchemy import create_engine, Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from datetime import datetime, timedelta
import os

# If DB_PATH not defined (user skipped Cell 2), use ephemeral DB
try:
    DB_PATH
except NameError:
    DB_PATH = '/content/library.db'
    print("DB_PATH not found — using ephemeral:", DB_PATH)

# Ensure folder exists
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

Base = declarative_base()

class Book(Base):
    __tablename__ = "book"
    id = Column(Integer, primary_key=True)
    title = Column(String(256), nullable=False)
    author = Column(String(128))
    isbn = Column(String(32), unique=True, index=True)
    total_copies = Column(Integer, nullable=False, default=1)
    available_copies = Column(Integer, nullable=False, default=1)
    published_year = Column(Integer)
    category = Column(String(64))
    created_at = Column(DateTime, default=datetime.utcnow)
    loans = relationship("Loan", back_populates="book", cascade="all, delete-orphan")

class Member(Base):
    __tablename__ = "member"
    id = Column(Integer, primary_key=True)
    name = Column(String(128), nullable=False)
    email = Column(String(128), unique=True, index=True)
    phone = Column(String(32))
    joined_at = Column(DateTime, default=datetime.utcnow)
    loans = relationship("Loan", back_populates="member", cascade="all, delete-orphan")

class Loan(Base):
    __tablename__ = "loan"
    id = Column(Integer, primary_key=True)
    book_id = Column(Integer, ForeignKey("book.id"), nullable=False)
    member_id = Column(Integer, ForeignKey("member.id"), nullable=False)
    issued_at = Column(DateTime, default=datetime.utcnow)
    due_date = Column(DateTime)
    returned_at = Column(DateTime, nullable=True)
    status = Column(String(16), default="issued")
    book = relationship("Book", back_populates="loans")
    member = relationship("Member", back_populates="loans")

# create engine and session
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False, connect_args={"check_same_thread": False})
Session = sessionmaker(bind=engine)
Base.metadata.create_all(engine)
session = Session()

# Helper functions
def add_book(title, author=None, isbn=None, total_copies=1, published_year=None, category=None):
    existing = None
    if isbn:
        existing = session.query(Book).filter_by(isbn=isbn).first()
    if existing:
        raise Exception("Book with this ISBN already exists (id {}).".format(existing.id))
    book = Book(title=title, author=author, isbn=isbn,
                total_copies=total_copies, available_copies=total_copies,
                published_year=published_year, category=category)
    session.add(book)
    session.commit()
    return book

def list_books(q=None):
    qobj = session.query(Book)
    if q:
        like = f"%{q}%"
        qobj = qobj.filter((Book.title.ilike(like)) | (Book.author.ilike(like)) | (Book.isbn.ilike(like)))
    return qobj.order_by(Book.id).all()

def get_book(book_id):
    return session.query(Book).get(book_id)

def update_book(book_id, **kwargs):
    book = get_book(book_id)
    if not book: return None
    if "total_copies" in kwargs:
        new_total = int(kwargs["total_copies"])
        diff = new_total - book.total_copies
        book.total_copies = new_total
        book.available_copies = max(0, book.available_copies + diff)
    for k,v in kwargs.items():
        if hasattr(book, k) and k != "total_copies":
            setattr(book, k, v)
    session.commit()
    return book

def delete_book(book_id):
    book = get_book(book_id)
    if not book: return False
    issued = session.query(Loan).filter_by(book_id=book.id, status="issued").count()
    if issued > 0:
        raise Exception("Cannot delete: copies currently issued.")
    session.delete(book)
    session.commit()
    return True

def add_member(name, email=None, phone=None):
    existing = None
    if email:
        existing = session.query(Member).filter_by(email=email).first()
    if existing:
        raise Exception("Member with this email already exists (id {}).".format(existing.id))
    member = Member(name=name, email=email, phone=phone)
    session.add(member)
    session.commit()
    return member

def list_members():
    return session.query(Member).order_by(Member.id).all()

def get_member(member_id):
    return session.query(Member).get(member_id)

def issue_book(book_id, member_id, days=14):
    book = get_book(book_id)
    member = get_member(member_id)
    if not book: raise Exception("Book not found")
    if not member: raise Exception("Member not found")
    if book.available_copies < 1:
        raise Exception("No copies available")
    due = datetime.utcnow() + timedelta(days=days)
    loan = Loan(book=book, member=member, due_date=due, status="issued")
    book.available_copies -= 1
    session.add(loan)
    session.commit()
    return loan

def return_loan(loan_id):
    loan = session.query(Loan).get(loan_id)
    if not loan: raise Exception("Loan not found")
    if loan.status != "issued": raise Exception("Loan not in issued state")
    loan.returned_at = datetime.utcnow()
    loan.status = "returned"
    loan.book.available_copies = min(loan.book.total_copies, loan.book.available_copies + 1)
    session.commit()
    return loan

def list_loans(active_only=False):
    q = session.query(Loan)
    if active_only:
        q = q.filter(Loan.status=="issued")
    return q.order_by(Loan.issued_at.desc()).all()

# Nice repr helpers for interactive printing
def book_to_dict(b):
    return {"id": b.id, "title": b.title, "author": b.author, "isbn": b.isbn,
            "total_copies": b.total_copies, "available_copies": b.available_copies}

def member_to_dict(m):
    return {"id": m.id, "name": m.name, "email": m.email}

def loan_to_dict(l):
    return {"id": l.id, "book_id": l.book_id, "book_title": l.book.title,
            "member_id": l.member_id, "member_name": l.member.name,
            "issued_at": l.issued_at.isoformat(), "due_date": l.due_date.isoformat() if l.due_date else None,
            "status": l.status}


In [14]:

# --- Books ---
try:
    b1 = add_book(
        "Introduction to Modern Physics",
        "Arthur Beiser",
        isbn="1111111111",
        total_copies=3
    )
except Exception as e:
    print("Add book skipped:", e)
    b1 = session.query(Book).filter_by(isbn="1111111111").first()

try:
    b2 = add_book(
        "Introduction to Quantum Mechanics",
        "D.J. Griffiths",
        isbn="2222222222",
        total_copies=2
    )
except Exception as e:
    print("Add book skipped:", e)
    b2 = session.query(Book).filter_by(isbn="2222222222").first()

# --- Members ---
try:
    m1 = add_member("Arpita Mishra", "arpita@kmc.com")
except Exception as e:
    print("Add member skipped:", e)
    m1 = session.query(Member).filter_by(email="arpita@kmc.com").first()

try:
    m2 = add_member("Riya Sharma", "riya@kmc.com")
except Exception as e:
    print("Add member skipped:", e)
    m2 = session.query(Member).filter_by(email="riya@kmc.com").first()

# --- Issue book to Arpita ---
loan1 = issue_book(b1.id, m1.id, days=21)
print("Issued loan id:", loan1.id)

print("\nBooks:")
for b in list_books():
    print(book_to_dict(b))

print("\nMembers:")
for m in list_members():
    print(member_to_dict(m))

print("\nActive loans:")
for ln in list_loans(active_only=True):
    print(loan_to_dict(ln))


Add book skipped: Book with this ISBN already exists (id 3).
Add book skipped: Book with this ISBN already exists (id 4).
Add member skipped: Member with this email already exists (id 3).
Add member skipped: Member with this email already exists (id 4).
Issued loan id: 5

Books:
{'id': 1, 'title': 'SICP', 'author': 'Harold Abelson & Gerald Jay Sussman', 'isbn': '0262010771', 'total_copies': 5, 'available_copies': 2}
{'id': 2, 'title': 'Introduction to Algorithms', 'author': 'CLRS', 'isbn': '0262033844', 'total_copies': 2, 'available_copies': 2}
{'id': 3, 'title': 'Introduction to Modern Physics', 'author': 'Arthur Beiser', 'isbn': '1111111111', 'total_copies': 3, 'available_copies': 2}
{'id': 4, 'title': 'Introduction to Quantum Mechanics', 'author': 'D.J. Griffiths', 'isbn': '2222222222', 'total_copies': 2, 'available_copies': 2}

Members:
{'id': 1, 'name': 'John Doe', 'email': 'john@example.com'}
{'id': 2, 'name': 'Priya Sharma', 'email': 'priya@example.com'}
{'id': 3, 'name': 'Arpit

/tmp/ipython-input-84378332.py:131: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  due = datetime.utcnow() + timedelta(days=days)


After running Cell 4
1. Books were added
2. Members were added
3. Loan (issue) was successful
4. Database is working perfectly

In [9]:

loan_id = loan1.id
print("Returning loan:", loan_id)
return_loan(loan_id)
print("\nAfter return - Books:")
for b in list_books():
    print(book_to_dict(b))

print("\nLoans (all):")
for ln in list_loans():
    print(loan_to_dict(ln))


Returning loan: 4

After return - Books:
{'id': 1, 'title': 'SICP', 'author': 'Harold Abelson & Gerald Jay Sussman', 'isbn': '0262010771', 'total_copies': 3, 'available_copies': 0}
{'id': 2, 'title': 'Introduction to Algorithms', 'author': 'CLRS', 'isbn': '0262033844', 'total_copies': 2, 'available_copies': 2}
{'id': 3, 'title': 'Introduction to Modern Physics', 'author': 'Arthur Beiser', 'isbn': '1111111111', 'total_copies': 3, 'available_copies': 3}
{'id': 4, 'title': 'Introduction to Quantum Mechanics', 'author': 'D.J. Griffiths', 'isbn': '2222222222', 'total_copies': 2, 'available_copies': 2}

Loans (all):
{'id': 4, 'book_id': 3, 'book_title': 'Introduction to Modern Physics', 'member_id': 3, 'member_name': 'Arpita Misra', 'issued_at': '2025-12-06T10:18:19.640623', 'due_date': '2025-12-27T10:18:19.633002', 'status': 'returned'}
{'id': 3, 'book_id': 1, 'book_title': 'SICP', 'member_id': 3, 'member_name': 'Arpita Misra', 'issued_at': '2025-12-06T10:16:35.353490', 'due_date': '2025-12

/tmp/ipython-input-84378332.py:142: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  loan.returned_at = datetime.utcnow()


In [10]:

print("Search: 'Algorithms'")
for b in list_books(q="Algorithms"):
    print(book_to_dict(b))

# Update book total copies
print("\nUpdating total copies of book id 1 to 5")
update_book(1, total_copies=5)
print(get_book(1).total_copies, get_book(1).available_copies)


Search: 'Algorithms'
{'id': 2, 'title': 'Introduction to Algorithms', 'author': 'CLRS', 'isbn': '0262033844', 'total_copies': 2, 'available_copies': 2}

Updating total copies of book id 1 to 5
5 2


In [12]:
from google.colab import files
files.download(DB_PATH)   # downloads the sqlite file to your local machine


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
#MENU for the library management system
def library_menu():
    while True:
        print("\n===== Library Management System =====")
        print("1. Add Book")
        print("2. Add Member")
        print("3. Issue Book")
        print("4. Return Book")
        print("5. View All Books")
        print("6. View All Members")
        print("7. View Active Loans")
        print("8. Exit")

        choice = input("\nEnter your choice: ")

        # 1. Add Book
        if choice == "1":
            title = input("Book Title: ")
            author = input("Author: ")
            isbn = input("ISBN: ")
            total = int(input("Total Copies: "))
            try:
                book = add_book(title, author, isbn, total)
                print("Book added:", book_to_dict(book))
            except Exception as e:
                print("Error:", e)

        # 2. Add Member
        elif choice == "2":
            name = input("Member Name: ")
            email = input("Email: ")
            phone = input("Phone (optional): ")
            try:
                member = add_member(name, email, phone)
                print("Member added:", member_to_dict(member))
            except Exception as e:
                print("Error:", e)

        # 3. Issue Book
        elif choice == "3":
            bid = int(input("Book ID: "))
            mid = int(input("Member ID: "))
            try:
                loan = issue_book(bid, mid)
                print("Loan issued:", loan_to_dict(loan))
            except Exception as e:
                print("Error:", e)

        # 4. Return Book
        elif choice == "4":
            lid = int(input("Loan ID: "))
            try:
                loan = return_loan(lid)
                print("Loan returned:", loan_to_dict(loan))
            except Exception as e:
                print("Error:", e)

        # 5. View All Books
        elif choice == "5":
            print("\n=== Books List ===")
            for b in list_books():
                print(book_to_dict(b))

        # 6. View All Members
        elif choice == "6":
            print("\n=== Members List ===")
            for m in list_members():
                print(member_to_dict(m))

        # 7. View Active Loans
        elif choice == "7":
            print("\n=== Active Loans ===")
            for l in list_loans(active_only=True):
                print(loan_to_dict(l))

        # 8. Exit
        elif choice == "8":
            print("Exiting Library System...")
            break

        else:
            print("Invalid choice. Try again.")


In [ ]:
library_menu()



===== Library Management System =====
1. Add Book
2. Add Member
3. Issue Book
4. Return Book
5. View All Books
6. View All Members
7. View Active Loans
8. Exit

=== Books List ===
{'id': 1, 'title': 'SICP', 'author': 'Harold Abelson & Gerald Jay Sussman', 'isbn': '0262010771', 'total_copies': 5, 'available_copies': 2}
{'id': 2, 'title': 'Introduction to Algorithms', 'author': 'CLRS', 'isbn': '0262033844', 'total_copies': 2, 'available_copies': 2}
{'id': 3, 'title': 'Introduction to Modern Physics', 'author': 'Arthur Beiser', 'isbn': '1111111111', 'total_copies': 3, 'available_copies': 2}
{'id': 4, 'title': 'Introduction to Quantum Mechanics', 'author': 'D.J. Griffiths', 'isbn': '2222222222', 'total_copies': 2, 'available_copies': 2}

===== Library Management System =====
1. Add Book
2. Add Member
3. Issue Book
4. Return Book
5. View All Books
6. View All Members
7. View Active Loans
8. Exit
